# SymPy Background

We have been stuck on the linearization of our equations of motion for a long time. What we would like to do is to use the _SymPy_ module from python to do some of the heavy lifting for us, and then we can verify its correctness manually. To do this, first we want to perform the parametric linearization on a simpler problem, and confirm that we know how to go from the nonlinear equations to the linearized ones at various trim points.

## Simpler problem
The simpler problem we want to solve is
$$
\ddot{x} + 2\dot{x}^3 + 2x = 0
$$

Which we can rearrange into the state space equation
\begin{align}
\dot{x}_1 &= (x_1 + x_2)^2 \\
\dot{x}_2 &= (x_1 + 1)(x_2 + 2) + u
\end{align}

This system has equilibrium points at 
$$
x_2 = 0
$$
$$
u = 2x_1
$$

In [2]:
from sympy import *
x1, x2, t, u = symbols("x1 x2 t u")

In [3]:
f1 =Function("f1")
f2 = Function("f2")

In [4]:
f1 = x2
f2 = -2*x1 - 2*x2**3 + u

In [5]:
f1

x2

In [6]:
f2

u - 2*x1 - 2*x2**3

In [7]:
df1dx1 = f1.diff(x1)
df1dx1

0

In [8]:
df1dx2 = f1.diff(x2)
df1dx2

1

In [9]:
df1du = f1.diff(u)
df1du

0

In [10]:
df2dx1 = f2.diff(x1)
df2dx1

-2

In [11]:
df2dx2 = f2.diff(x2)
df2dx2

-6*x2**2

In [12]:
df2du = f2.diff(u)
df2du

1

In [13]:
A = Matrix([[f.diff(x) for x in [x1, x2]] for f in [f1, f2]])
A

Matrix([
[ 0,        1],
[-2, -6*x2**2]])

In [14]:
A.subs(x2, 0)

Matrix([
[ 0, 1],
[-2, 0]])

In [15]:
B = Matrix([[f.diff(u) for u in [u]] for f in [f1, f2]])
B

Matrix([
[0],
[1]])

# Applying Our system
We now want to turn to the system of equations we have interest in, which are much more complex than the ones we were just considering. First we need to define our variables.

In [16]:
# Shortcut functions for sin and cosine
x, y, z = symbols('x y z')
c = Lambda(x, cos(x))
s = Lambda(x, sin(x))

## Variable Definitions

Position Variable $\vec{p}_{B/N}^N$

In [17]:
pn, pe, pd = symbols("p_n p_e p_d")
pos = Matrix([[pn], [pe], [pd]])
pos

Matrix([
[p_n],
[p_e],
[p_d]])

Orientation $\vec{\Phi}_{B/N}^N$

In [18]:
phi, tht, psi = symbols("\\phi \\theta \\psi")
PHI = Matrix([[phi], [tht], [psi]])
PHI

Matrix([
[  \phi],
[\theta],
[  \psi]])

Velocity $\vec{v}_{B/E}^B$

In [19]:
u, v, w = symbols("u v w")
vel = Matrix([[u], [v], [w]])
vel

Matrix([
[u],
[v],
[w]])

Angular Velocity $\vec{\omega}_{B/E}^B$

In [20]:
p, q, r = symbols("P Q R")
omg = Matrix([[p], [q], [r]])
omg

Matrix([
[P],
[Q],
[R]])

## Dynamic Derivative Definitions

$\dot{\vec{p}}_{B/N}^N$

In [21]:
C_bfn = Matrix([[c(tht)*c(psi), c(tht)*s(psi), -s(tht)],
                [-c(phi)*s(psi)+s(phi)*s(tht)*c(psi), c(phi)*c(psi)+s(phi)*s(tht)*s(psi), s(phi)*c(tht)],
                [s(phi)*s(psi)+c(phi)*s(tht)*c(psi), -s(phi)*c(psi)+c(phi)*s(tht)*s(psi), c(phi)*c(tht)]])
C_bfn

Matrix([
[                                cos(\psi)*cos(\theta),                                  sin(\psi)*cos(\theta),          -sin(\theta)],
[sin(\phi)*sin(\theta)*cos(\psi) - sin(\psi)*cos(\phi),  sin(\phi)*sin(\psi)*sin(\theta) + cos(\phi)*cos(\psi), sin(\phi)*cos(\theta)],
[sin(\phi)*sin(\psi) + sin(\theta)*cos(\phi)*cos(\psi), -sin(\phi)*cos(\psi) + sin(\psi)*sin(\theta)*cos(\phi), cos(\phi)*cos(\theta)]])

In [22]:
dpos_brn_n = C_bfn.T*vel
dpos_brn_n

Matrix([
[ u*cos(\psi)*cos(\theta) + v*(sin(\phi)*sin(\theta)*cos(\psi) - sin(\psi)*cos(\phi)) + w*(sin(\phi)*sin(\psi) + sin(\theta)*cos(\phi)*cos(\psi))],
[u*sin(\psi)*cos(\theta) + v*(sin(\phi)*sin(\psi)*sin(\theta) + cos(\phi)*cos(\psi)) + w*(-sin(\phi)*cos(\psi) + sin(\psi)*sin(\theta)*cos(\phi))],
[                                                                              -u*sin(\theta) + v*sin(\phi)*cos(\theta) + w*cos(\phi)*cos(\theta)]])

$\dot{\vec{\Phi}}_{B/N}^N$

In [23]:
H_phi = Matrix([[1, s(phi)*tan(tht), c(phi)*tan(tht)],
                [0, c(phi), -s(phi)],
                [0, s(phi)/c(tht), c(phi)/c(tht)]])
H_phi

Matrix([
[1, sin(\phi)*tan(\theta), cos(\phi)*tan(\theta)],
[0,             cos(\phi),            -sin(\phi)],
[0, sin(\phi)/cos(\theta), cos(\phi)/cos(\theta)]])

In [24]:
dPHI_brn_n = H_phi*omg
dPHI_brn_n

Matrix([
[P + Q*sin(\phi)*tan(\theta) + R*cos(\phi)*tan(\theta)],
[                            Q*cos(\phi) - R*sin(\phi)],
[    Q*sin(\phi)/cos(\theta) + R*cos(\phi)/cos(\theta)]])

$\dot{\vec{v}}_{B/E}^B$

In [25]:
m, g = symbols("m g")
gvec = Matrix([[0],[0],[g]])
C_bfn*gvec

Matrix([
[         -g*sin(\theta)],
[g*sin(\phi)*cos(\theta)],
[g*cos(\phi)*cos(\theta)]])

In [26]:
omg.cross(vel)

Matrix([
[ Q*w - R*v],
[-P*w + R*u],
[ P*v - Q*u]])

In [27]:
wn, we, wd = symbols("w_n w_e w_d")
v_wre_e = Matrix([[wn],[we],[wd]])
v_rel = vel - C_bfn*v_wre_e
v_rel

Matrix([
[                                                                               u + w_d*sin(\theta) - w_e*sin(\psi)*cos(\theta) - w_n*cos(\psi)*cos(\theta)],
[ v - w_d*sin(\phi)*cos(\theta) - w_e*(sin(\phi)*sin(\psi)*sin(\theta) + cos(\phi)*cos(\psi)) - w_n*(sin(\phi)*sin(\theta)*cos(\psi) - sin(\psi)*cos(\phi))],
[w - w_d*cos(\phi)*cos(\theta) - w_e*(-sin(\phi)*cos(\psi) + sin(\psi)*sin(\theta)*cos(\phi)) - w_n*(sin(\phi)*sin(\psi) + sin(\theta)*cos(\phi)*cos(\psi))]])

In [28]:
V_T = v_rel.norm()

In [29]:
alpha = atan2(v_rel[2], v_rel[0])

In [30]:
beta = asin(v_rel[1]/V_T)

In [31]:
alpha

atan2(w - w_d*cos(\phi)*cos(\theta) - w_e*(-sin(\phi)*cos(\psi) + sin(\psi)*sin(\theta)*cos(\phi)) - w_n*(sin(\phi)*sin(\psi) + sin(\theta)*cos(\phi)*cos(\psi)), u + w_d*sin(\theta) - w_e*sin(\psi)*cos(\theta) - w_n*cos(\psi)*cos(\theta))

In [32]:
C_wfb = Matrix([[c(alpha)*c(beta), s(beta), s(alpha)*c(beta)],
                [-c(alpha)*s(beta), c(beta), -s(alpha)*s(beta)],
                [-s(alpha), 0, c(alpha)]])

In [33]:
rho = symbols("\\rho")

Define Wing features

In [34]:
ail1, ail2, elev, rudd = symbols("\\delta{}a_1 \\delta{}a_2 \\delta{}e \\delta{}r")
i1, i2, i3, i4 = symbols("i_1 i_2 i_3 i_4")
xs1, ys1, zs1 = symbols("x_{s1} y_{s1} z_{s1}")
xvecs1 = Matrix([[xs1],[ys1],[zs1]])
xp1, yp1, zp1 = symbols("x_{p1} y_{p1} z_{p1}")
xvecp1 = Matrix([[xp1],[yp1],[zp1]])


Front left wing

In [35]:
b1, c1 = symbols("b_1 c_1")
S1 = b1*c1
AR1 = b1/c1
Cl01, dCldu1, cu1 = symbols("C_{L01} \\frac{\\partial{}C_{L1}}{\\partial\\delta{}u} c_{u1}")
Cla1 = 2*pi*AR1/(2+AR1)
CL1 = Lambda((x, y), Cl01 + Cla1*x + dCldu1*y*cu1/c1)
alpha_s1 = alpha + i1

In [36]:
CL1(x, y)

C_{L01} + \frac{\partial{}C_{L1}}{\partial\delta{}u}*c_{u1}*y/c_1 + 2*pi*b_1*x/(c_1*(b_1/c_1 + 2))

In [37]:
L1 = 1/2*rho*v_rel.dot(v_rel)*S1*CL1(alpha_s1, ail1)

In [38]:
Cd01, Cda1, a01, e1, dCddu1 = symbols("C_{D01} C_{d\\alpha{}1} \\alpha_{01} e_1 \\frac{\\partial{}C_{D1}}{\\partial\\delta{}u_1}")
CD1 = Lambda((x, y), Cd01 + Cda1*(x-a01)**2 + CL1(x, y)**2/pi/e1/AR1 + dCddu1*y*cu1/c1)

In [39]:
CD1(x, y)

C_{D01} + C_{d\alpha{}1}*(-\alpha_{01} + x)**2 + \frac{\partial{}C_{D1}}{\partial\delta{}u_1}*c_{u1}*y/c_1 + c_1*(C_{L01} + \frac{\partial{}C_{L1}}{\partial\delta{}u}*c_{u1}*y/c_1 + 2*pi*b_1*x/(c_1*(b_1/c_1 + 2)))**2/(pi*b_1*e_1)

In [40]:
D1 = 1/2*rho*v_rel.dot(v_rel)*S1*CD1(alpha_s1, ail1)

In [41]:
F_a1_w = Matrix([[-D1],[0],[-L1]])

In [42]:
F_a1_b = C_wfb.T*F_a1_w
F_a1_b

Matrix([
[-0.5*\rho*b_1*c_1*sqrt(1 - (v - w_d*sin(\phi)*cos(\theta) - w_e*(sin(\phi)*sin(\psi)*sin(\theta) + cos(\phi)*cos(\psi)) - w_n*(sin(\phi)*sin(\theta)*cos(\psi) - sin(\psi)*cos(\phi)))**2/(Abs(u + w_d*sin(\theta) - w_e*sin(\psi)*cos(\theta) - w_n*cos(\psi)*cos(\theta))**2 + Abs(-v + w_d*sin(\phi)*cos(\theta) + w_e*(sin(\phi)*sin(\psi)*sin(\theta) + cos(\phi)*cos(\psi)) + w_n*(sin(\phi)*sin(\theta)*cos(\psi) - sin(\psi)*cos(\phi)))**2 + Abs(w - w_d*cos(\phi)*cos(\theta) - w_e*(-sin(\phi)*cos(\psi) + sin(\psi)*sin(\theta)*cos(\phi)) - w_n*(sin(\phi)*sin(\psi) + sin(\theta)*cos(\phi)*cos(\psi)))**2))*((u + w_d*sin(\theta) - w_e*sin(\psi)*cos(\theta) - w_n*cos(\psi)*cos(\theta))**2 + (v - w_d*sin(\phi)*cos(\theta) - w_e*(sin(\phi)*sin(\psi)*sin(\theta) + cos(\phi)*cos(\psi)) - w_n*(sin(\phi)*sin(\theta)*cos(\psi) - sin(\psi)*cos(\phi)))**2 + (w - w_d*cos(\phi)*cos(\theta) - w_e*(-sin(\phi)*cos(\psi) + sin(\psi)*sin(\theta)*cos(\phi)) - w_n*(sin(\phi)*sin(\psi) + sin(\theta)*cos(\ph

In [43]:
Thp1, tau1 = symbols("T_{p1} \\tau_1")
n1n, n1e, n1d = symbols("n_{1N} n_{1E} n_{1D}")
n_p1 = Matrix([[n1n],[n1e],[n1d]])
s_p1 = symbols("s_1")

In [44]:
F_p1_b = Thp1*n_p1

In [45]:
M_p1_b = tau1*s_p1*n_p1

Aside:
The motors can and should be described separately, using the following equations of motion
\begin{align}
\tau &= \frac{K_T}{R_m}(T - K_v\omega_m \\
\dot{\omega}_m &= \frac{-K_TK_V}{R_mJ_m}\omega_m - \frac{C_D}{J_m}\omega_m^2 + \frac{K_T}{R_mJ_m}T \\
T_p &= C_{Lm}\omega^2
\end{align}
With $\omega_m$ as the rotational speed of the motor, $\tau$ the torque generated by the motor, $T$ the requested thrust input to the motor system, and $T$ the actual thrust output from the motor system. For now, we will just model the relevant "outputs" of $\tau$ and $T_p$

In [46]:
Cm01, Cma1, dCmdu1 = symbols("C_{M01} C_{M\\alpha{}1} \\frac{\\partial{}C_M1}{\\partial\\delta{}u_1}") 

In [47]:
CM1 = Lambda((x, y), Cm01 + Cma1*x + dCmdu1*y*cu1/c1)

In [48]:
M1 = 1/2*rho*S1*c1*v_rel.dot(v_rel)*CM1(alpha_s1,ail1)

In [49]:
M_a1_b = Matrix([[0],[M1],[0]])

In [50]:
M_f1_b = xvecs1.cross(F_a1_b)

In [51]:
M_T1_b = xvecp1.cross(F_p1_b)

In [52]:
F_ext_b = (F_a1_b) + (F_p1_b)

In [53]:
M_ext_b = (M_a1_b + M_f1_b) + (M_T1_b + M_p1_b)

$\dot{\vec{v}}_{B/E}^B$

In [54]:
dvel_bre_b = 1/m*F_ext_b + C_bfn*gvec - omg.cross(vel)

$\dot{\vec{\omega}}_{B/E}^B$

In [55]:
Ixx, Iyy, Izz, Ixy, Iyz, Ixz = symbols("I_{xx} I_{yy} I_{zz} I_{xy} I_{yz} I_{xz}")
J = Matrix([[Ixx, Ixy, Ixz],
            [Ixy, Iyy, Iyz],
            [Ixz, Iyz, Izz]])

In [56]:
domg_bre_b = J.inv()*(M_ext_b - omg.cross(J*omg))

## Linearization

We now have 12 state variables $(p_N, p_E, p_D, \phi, \theta, \psi, u, v, w, p, q, r)$ and a certain number of input variables, which for now since we only defined one wing and one propeller is $(a, e, r, T_p, \tau, w_N, w_E, w_D)$. We want to take each of our equations and linearize them against these variables, to get the linearized equations.

In [57]:
dpos_brn_n.diff(pos)

[[[[0], [0], [0]]], [[[0], [0], [0]]], [[[0], [0], [0]]]]

In [75]:
dpos_brn_n.subs({tht: 0, psi: 0}).diff(phi)

Matrix([
[                         0],
[-v*sin(\phi) - w*cos(\phi)],
[ v*cos(\phi) - w*sin(\phi)]])

In [76]:
dpos_brn_n.subs({phi: 0, psi: 0}).diff(tht)

Matrix([
[-u*sin(\theta) + w*cos(\theta)],
[                             0],
[-u*cos(\theta) - w*sin(\theta)]])

In [82]:
dpos_brn_n.subs({phi: pi/6, tht: pi/4, psi: 0}).diff(vel)

[[[[sqrt(2)/2], [0], [-sqrt(2)/2]]], [[[sqrt(2)/4], [sqrt(3)/2], [sqrt(2)/4]]], [[[sqrt(6)/4], [-1/2], [sqrt(6)/4]]]]

In [58]:
dpos_brn_n.diff(PHI)

[[[[v*(sin(\phi)*sin(\psi) + sin(\theta)*cos(\phi)*cos(\psi)) + w*(-sin(\phi)*sin(\theta)*cos(\psi) + sin(\psi)*cos(\phi))], [v*(-sin(\phi)*cos(\psi) + sin(\psi)*sin(\theta)*cos(\phi)) + w*(-sin(\phi)*sin(\psi)*sin(\theta) - cos(\phi)*cos(\psi))], [v*cos(\phi)*cos(\theta) - w*sin(\phi)*cos(\theta)]]], [[[-u*sin(\theta)*cos(\psi) + v*sin(\phi)*cos(\psi)*cos(\theta) + w*cos(\phi)*cos(\psi)*cos(\theta)], [-u*sin(\psi)*sin(\theta) + v*sin(\phi)*sin(\psi)*cos(\theta) + w*sin(\psi)*cos(\phi)*cos(\theta)], [-u*cos(\theta) - v*sin(\phi)*sin(\theta) - w*sin(\theta)*cos(\phi)]]], [[[-u*sin(\psi)*cos(\theta) + v*(-sin(\phi)*sin(\psi)*sin(\theta) - cos(\phi)*cos(\psi)) + w*(sin(\phi)*cos(\psi) - sin(\psi)*sin(\theta)*cos(\phi))], [u*cos(\psi)*cos(\theta) + v*(sin(\phi)*sin(\theta)*cos(\psi) - sin(\psi)*cos(\phi)) + w*(sin(\phi)*sin(\psi) + sin(\theta)*cos(\phi)*cos(\psi))], [0]]]]

In [59]:
dpos_brn_n.diff(vel)

[[[[cos(\psi)*cos(\theta)], [sin(\psi)*cos(\theta)], [-sin(\theta)]]], [[[sin(\phi)*sin(\theta)*cos(\psi) - sin(\psi)*cos(\phi)], [sin(\phi)*sin(\psi)*sin(\theta) + cos(\phi)*cos(\psi)], [sin(\phi)*cos(\theta)]]], [[[sin(\phi)*sin(\psi) + sin(\theta)*cos(\phi)*cos(\psi)], [-sin(\phi)*cos(\psi) + sin(\psi)*sin(\theta)*cos(\phi)], [cos(\phi)*cos(\theta)]]]]

In [60]:
dpos_brn_n.diff(omg)

[[[[0], [0], [0]]], [[[0], [0], [0]]], [[[0], [0], [0]]]]

In [61]:
surfaces = Matrix([[ail1], [ail2], [elev], [rudd]])

In [62]:
props = Matrix([[Thp1], [tau1]])

In [63]:
wind = Matrix([[wn],[we],[wd]])

In [64]:
dpos_brn_n.diff(surfaces)

[[[[0], [0], [0]]], [[[0], [0], [0]]], [[[0], [0], [0]]], [[[0], [0], [0]]]]

In [65]:
dpos_brn_n.diff(props)

[[[[0], [0], [0]]], [[[0], [0], [0]]]]

In [66]:
dpos_brn_n.diff(wind)

[[[[0], [0], [0]]], [[[0], [0], [0]]], [[[0], [0], [0]]]]

In [67]:
dPHI_brn_n.diff(pos)

[[[[0], [0], [0]]], [[[0], [0], [0]]], [[[0], [0], [0]]]]

In [68]:
dPHI_brn_n.diff(PHI)

[[[[Q*cos(\phi)*tan(\theta) - R*sin(\phi)*tan(\theta)], [-Q*sin(\phi) - R*cos(\phi)], [Q*cos(\phi)/cos(\theta) - R*sin(\phi)/cos(\theta)]]], [[[Q*(tan(\theta)**2 + 1)*sin(\phi) + R*(tan(\theta)**2 + 1)*cos(\phi)], [0], [Q*sin(\phi)*sin(\theta)/cos(\theta)**2 + R*sin(\theta)*cos(\phi)/cos(\theta)**2]]], [[[0], [0], [0]]]]

In [ ]:
dPHI_brn_n.diff(vel)

In [ ]:
dPHI_brn_n.diff(omg)

In [ ]:
dPHI_brn_n.diff(surfaces)

In [ ]:
dPHI_brn_n.diff(props)

In [ ]:
dPHI_brn_n.diff(wind)

In [ ]:
dvel_bre_b.diff(pos)

In [ ]:
dvel_bre_b.diff(PHI)

In [ ]:
dvel_bre_b.diff(vel)

In [ ]:
dvel_bre_b.diff(omg)

In [ ]:
dvel_bre_b.diff(surfaces)

In [ ]:
dvel_bre_b.diff(props)

In [ ]:
dvel_bre_b.diff(wind)

In [ ]:
domg_bre_b.diff(pos)

In [ ]:
domg_bre_b.diff(PHI)

In [ ]:
domg_bre_b.diff(vel)

In [ ]:
domg_bre_b.diff(omg)

In [ ]:
domg_bre_b.diff(surfaces)

In [ ]:
domg_bre_b.diff(props)

In [ ]:
domg_bre_b.diff(wind)